# Prerequisites

In [0]:
from pyspark.sql.functions import *
# customers_df = spark.read.table("ecommerce.silver.customers")
# orders_df = spark.read.table("ecommerce.silver.orders")
# order_items_df = spark.read.table("ecommerce.silver.order_items")
products_df = spark.read.table("ecommerce.silver.products")
# payments_df = spark.read.table("ecommerce.silver.payments")
# inventory_df = spark.read.table("ecommerce.silver.inventory")
clickstream_df = spark.read.table("ecommerce.silver.clickstream")

# Calculations

In [0]:
clickstream_df.limit(5).display()

## product_views, add_to_cart_count, checkout_count, purchase_count

In [0]:
prod_df = clickstream_df.groupBy("product_id").agg(
    sum(when(col("event_type") == "PRODUCT_VIEW", 1).otherwise(0)).alias(
        "product_views"
    ),
    sum(when(col("event_type") == "ADD_TO_CART", 1).otherwise(0)).alias(
        "add_to_cart_count"
    ),
    sum(when(col("event_type") == "CHECKOUT", 1).otherwise(0)).alias(
        "checkout_count"
    ),
    sum(when(col("event_type") == "PURCHASE", 1).otherwise(0)).alias(
        "purchase_count"
    ),
)

prod_df.limit(5).display()

## view_to_cart_rate, cart_to_purchase_rate, overall_conversion_rate

In [0]:
product_conversion_df = (
    prod_df.withColumn(
        "view_to_cart_rate",
        round(try_divide(col("add_to_cart_count"), col("product_views")), 2),
    )
    .withColumn(
        "cart_to_purchase_rate",
        round(try_divide(col("purchase_count"), col("add_to_cart_count")), 2),
    )
    .withColumn(
        "overall_conversion_rate",
        round(try_divide(col("purchase_count"), col("product_views")), 2),
    )
)
product_conversion_df.limit(5).display()

# Create gold.product_conversion Delta table

In [0]:
product_conversion_df.write.mode("overwrite").option("mergeSchema","true").saveAsTable("ecommerce.gold.product_conversion")
spark.read.table("ecommerce.gold.product_conversion").limit(5).display()

# Business Question

## Which products have high customer interest but low purchase conversion?

In [0]:
spark.read.table("ecommerce.gold.product_conversion").select(
    min(col("product_views")).alias("mini"),
    max(col("product_views")).alias("maxi"),
    min(col("overall_conversion_rate")).alias("mini_c"),
    max(col("overall_conversion_rate")).alias("maxi_c")
).display()

In [0]:
spark.read.table("ecommerce.gold.product_conversion").filter(
    (col("product_views") >= 50) & (col("overall_conversion_rate") <= 0.05)
).orderBy(col("product_views").desc()).join(products_df, "product_id").select(
    "product_name", "product_views"
).display()